### 00 Data Dictionary

In [3]:
from pathlib import Path
import pandas as pd
import requests
import openpyxl as xl

DOCS = Path("../docs/data_dictionary")   # adjust to your notebook's location
DOCS.mkdir(parents=True, exist_ok=True)

In [4]:
# 1. FEMA field definitions
FEMA_URL = "https://www.fema.gov/api/open/v1/OpenFemaDataSetFields"
FEMA_DATASETS = {
    "NfipPolicies": 3,
    "NfipClaims": 3,
    "DisasterDeclarationsSummaries": 2,
    "NfipCommunityStatusBook": 1,
}
COLUMNS = ["openFemaDataSet", "datasetVersion", "sortOrder", "name", "title",
           "type", "isNullable", "primaryKey", "description"]

frames = []
for dataset, version in FEMA_DATASETS.items():
    params = {
        "$filter": f"openFemaDataSet eq '{dataset}' and datasetVersion eq {version}",
        "$top": 1000,
        "$format": "json",
    }
    r = requests.get(FEMA_URL, params=params, timeout=120)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["OpenFemaDataSetFields"])[COLUMNS].sort_values("sortOrder")
    df.to_excel(DOCS / f"fema_{dataset}_v{version}.xlsx", index=False)
    print(f"{dataset}: {len(df)} fields")
    frames.append(df)

fema_dict = pd.concat(frames, ignore_index=True)

NfipPolicies: 87 fields
NfipClaims: 84 fields
DisasterDeclarationsSummaries: 29 fields
NfipCommunityStatusBook: 18 fields


In [5]:
# 2. Census ACS variables used in this project
ACS_VARS = ["B19013_001E", "B25077_001E", "B01002_001E",
            "B25003_001E", "B25003_002E", "B01003_001E"]

r = requests.get("https://api.census.gov/data/2022/acs/acs5/variables.json", timeout=300)
r.raise_for_status()
variables = r.json()["variables"]

acs_dict = pd.DataFrame(
    [{"name": v, "label": variables[v]["label"], "concept": variables[v]["concept"],
      "type": variables[v].get("predicateType")} for v in ACS_VARS]
)
acs_dict.to_excel(DOCS / "census_acs5_2022_selected.xlsx", index=False)

In [6]:
# 3. Show the cancellation reason codes
codes = fema_dict.query(
    "openFemaDataSet == 'NfipPolicies' and name == 'cancellationVoidanceReasonCode'"
)["description"].iloc[0]
print(codes)

Reason code indicating why a policy was cancelled/nullified.
Legal values (value : description):
01 : Building Sold or Removed;
02 : Contents Sold or Removed (contents-only policy);
03 : Policy Canceled and Rewritten to Establish a Common Expiration; Date with Other Lines of Insurance;
04 : Duplicate Policies (NFIP);
05 : Non-Payment;
06 : Ineligible Risk;
08 : No Insurable Interest;
09 : PMR (Physical Map Revision) or LOMR ( Letter of Map Revision) with Lender Release;
10 : Closed Basin Lake (ineligible);
16 : FEMA Letter of Determination with Lender Release;
17 : Duplicate Policy from Source Other than the NFIP (Legacy System);
20 : LOMA (Letter of Map Amendment) with Lender Release;
21 : SRL (Severe Repetitive Loss) Written with Incorrect Insurer;
22 : Cancel/Rewrite Due to Misrating;
23 : Fraud;
24 : Cancel/Rewrite to PRP after Map Revision;
25 : Cancel/Rewrite to Process HFIAA Refund;
26 : Duplicate Policy from Source Other than the NFIP;
27 : Property becomes ineligible for cover